In [ ]:
!pip install ARTSyn

In [ ]:
import pandas as pd
from artsyn.generators import ctd_gan
random_state = 42
from sklearn.preprocessing import LabelEncoder

In [ ]:
df1 = pd.read_csv("https://raw.githubusercontent.com/senihberkay/US-Army-ANSUR-II/master/ANSUR%20II%20FEMALE%20Public.csv", encoding='latin1')

In [ ]:
df2 = pd.read_csv("https://raw.githubusercontent.com/senihberkay/US-Army-ANSUR-II/master/ANSUR%20II%20MALE%20Public.csv", encoding='latin1')

In [ ]:
df1.head()

In [ ]:
df2.head()

In [ ]:
df2.rename(columns={'subjectid':'SubjectId'}, inplace=True)

In [ ]:
df1.columns == df2.columns

In [ ]:
print(df1.columns.tolist())

In [ ]:
df = pd.concat([df1, df2], ignore_index=True)
display(df.head())

In [ ]:
df.info()

In [ ]:
df.set_index('SubjectId', inplace=True)
display(df.head())

In [ ]:
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]
display(missing_values.sort_values(ascending=False))

In [ ]:
columns_to_drop = ['Date', 'Installation', 'Component', 'Branch', 'PrimaryMOS', 'SubjectsBirthLocation', 'SubjectNumericRace', 'Ethnicity', 'DODRace', 'Age', 'Heightin', 'Weightlbs', 'WritingPreference']
df.drop(columns=columns_to_drop, inplace=True)
display(df.head())

In [ ]:
duplicate_rows = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

In [ ]:
df.describe()

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(df, test_size=0.7, random_state=42, stratify=df['Gender'])

print(f"Shape of training set (df_train): {df_train.shape}")
print(f"Shape of validation set (df_val): {df_val.shape}")

### Separate features (X) and target (y)

In [ ]:
TARGET_COLUMN = 'Gender'

X_train = df_train.drop(columns=[TARGET_COLUMN])
y_train = df_train[TARGET_COLUMN]

X_val = df_val.drop(columns=[TARGET_COLUMN])
y_val = df_val[TARGET_COLUMN]

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")

In [ ]:
X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy()


In [ ]:



# 1. Xác định danh sách cột phân loại (categorical) nếu có.
# Nhập index của các cột phân loại trong X_train. Ví dụ: categorical_cols = (1, 3)
# Nếu tất cả các cột của bạn đều là dữ liệu số liên tục, hãy để tuple rỗng:
categorical_columns = ()
num_classes = 2
# 2. Khởi tạo đối tượng ctdGAN với các siêu tham số phù hợp
gan = ctd_gan.ctdGAN(
    discriminator=(256, 256),   # Kiến trúc mạng Critic (Discriminator)
    generator=(256, 256),       # Kiến trúc mạng Generator
    epochs=150,                 # Số chu kỳ huấn luyện đối kháng (khuyên dùng từ 100-300 epochs để tối ưu chất lượng)
    batch_size=100,             # Kích thước lô huấn luyện
    pac=10,                     # Đóng gói mẫu chống sụp đổ chế độ (mode collapse)
    embedding_dim=128,          # Số chiều của không gian ẩn (latent space)
    max_clusters=10,            # Số phân cụm tối đa trong không gian con thực tế
    cluster_method='kmeans',    # Thuật toán phân cụm không gian thực
    scaler='mms11',             # Kỹ thuật chuẩn hóa đưa dữ liệu về khoảng [-1, 1]
    sampling_strategy='create-new', # Chiến lược sinh mẫu điều kiện
    random_state=42             # Seed để đảm bảo kết quả có thể tái lặp
)


In [ ]:


# Encode y_train_np to numerical values
le = LabelEncoder()
y_train_encoded_np = le.fit_transform(y_train_np)

# 3. Huấn luyện mô hình và thực hiện sinh thêm mẫu (Oversampling)
# Hàm này sẽ học phân phối dữ liệu gốc và tự động sinh thêm dữ liệu mới
gan.fit(X_train_np, y_train_encoded_np)

In [ ]:
for index, label in enumerate(le.classes_):
    print(f"{index} -> {label}")

In [ ]:
male_generated = gan.sample(5000,1)

In [ ]:
female_generated = gan.sample(5000,0)

In [ ]:
male_generated

In [26]:
import numpy as np
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import rbf_kernel

# =====================================================================
# STEP 1: ĐỒNG BỘ HÓA DỮ LIỆU THẬT & GỘP NHÃN DỮ LIỆU TẠO RA (AI SYNTH)
# =====================================================================

# 1. Khôi phục nhãn chuỗi gốc cho dữ liệu giả từ chính cấu trúc mảng sinh của bạn
labels_male_synth = np.full(male_generated.shape[0], 'Male')
labels_female_synth = np.full(female_generated.shape[0], 'Female')

# Gộp ma trận đặc trưng (X) và mảng nhãn chuỗi (y) của tập dữ liệu ctdGAN
X_synth = np.vstack((male_generated, female_generated))
y_synth_str = np.concatenate((labels_male_synth, labels_female_synth))

# 2. Đồng bộ hóa nhãn số nguyên bằng đối tượng LabelEncoder (le) bạn đã dùng cho dữ liệu thật
# Điều này giúp y_synth_encoded tương thích hoàn toàn với y_train_encoded_np khi huấn luyện Downstream ML
y_synth_encoded = le.transform(y_synth_str)

# Ánh xạ các mảng dữ liệu gốc của bạn vào luồng xử lý chính
X_real = X_train_np
y_real_encoded = y_train_encoded_np

print("--- THÔNG SỐ ĐẦU VÀO HỆ THỐNG ---")
print(f" -> Kích thước tập dữ liệu THẬT (X_train_np): {X_real.shape}")
print(f" -> Kích thước tập dữ liệu GIẢ   (X_synth):    {X_synth.shape}")


# =====================================================================
# STEP 2: THUẬT TOÁN KIỂM ĐỊNH PHÂN PHỐI ĐÃ KHẮC PHỤC LỖI QUY MÔ (DATA SCALE)
# =====================================================================

def calculate_corrected_synos_metrics(X_real_np, X_synth_np):
    """
    Hàm tính toán khoảng cách phân phối đồng bộ thang đo.
    Sử dụng StandardScaler của dữ liệu THẬT để đưa cả hai tập về cùng phân phối (Mean=0, Std=1).
    """
    # 1. Tiến hành Z-score Scaling để triệt tiêu ảnh hưởng đơn vị đo thô (mm)
    scaler = StandardScaler()
    X_real_scaled = scaler.fit_transform(X_real_np)
    X_synth_scaled = scaler.transform(X_synth_np)

    # 2. Đo lường Wasserstein Distance đơn biến trên dữ liệu đã Chuẩn hóa
    num_features = X_real_scaled.shape[1]
    wd_scaled_list = []

    for i in range(num_features):
        wd = wasserstein_distance(X_real_scaled[:, i], X_synth_scaled[:, i])
        wd_scaled_list.append(wd)

    mean_wd_scaled = np.mean(wd_scaled_list)
    # Quy đổi khoảng cách chuẩn hóa sang tỉ lệ % độ tin cậy hình thái cột
    wd_percentage = np.exp(-mean_wd_scaled) * 100

    # 3. Đo lường MMD đa biến trên không gian phân phối gốc (RBF Kernel tự kiểm soát biên độ)
    gamma = 1.0 / X_real_np.shape[1]
    K_real_real = rbf_kernel(X_real_np, X_real_np, gamma=gamma)
    K_synth_synth = rbf_kernel(X_synth_np, X_synth_np, gamma=gamma)
    K_real_synth = rbf_kernel(X_real_np, X_synth_np, gamma=gamma)

    mmd_squared = K_real_real.mean() + K_synth_synth.mean() - 2 * K_real_synth.mean()
    raw_mmd = np.sqrt(max(0, mmd_squared))
    # Quy đổi khoảng cách đa biến sang tỉ lệ % độ tin cậy tương quan giải phẫu
    mmd_percentage = np.exp(-raw_mmd) * 100

    # 4. Tính điểm số tin cậy phân phối tổng hợp (Overall Score)
    overall_trust_score = (wd_percentage + mmd_percentage) / 2

    return mean_wd_scaled, wd_percentage, raw_mmd, mmd_percentage, overall_trust_score


# =====================================================================
# STEP 3: THỰC THI CHẤM ĐIỂM & ĐÓNG GÓI KẾT QUẢ BÁO CÁO NGHIÊN CỨU
# =====================================================================

# Chạy thuật toán xử lý đồng bộ
raw_wd, wd_pct, raw_mmd, mmd_pct, overall_score = calculate_corrected_synos_metrics(X_real, X_synth)

# Xuất bảng hiển thị kết quả chuẩn mực khoa học
print("\n" + "="*55)
print("   KẾT QUẢ KIỂM ĐỊNH PHÂN PHỐI DỮ LIỆU TỔNG HỢP HOÀN CHỈNH")
print("="*55)
print(f"1. Wasserstein Distance Metric (Đơn biến - Đã chuẩn hóa):")
print(f"   - Khoảng cách thô định lượng: {raw_wd:.4f}")
print(f"   - Độ tin cậy hình thái cột   : {wd_pct:.2f}%")
print(f"     (Tránh lỗi tràn số mũ của dữ liệu trắc lượng thô)\n")
print(f"2. Maximum Mean Discrepancy Metric (Đa biến toàn cục):")
print(f"   - Khoảng cách thô tương quan: {raw_mmd:.4f}")
print(f"   - Độ tin cậy tương quan giải phẫu (Allometry): {mmd_pct:.2f}%\n")
print("-" * 55)
print(f"--> OVERALL DISTRIBUTION TRUST SCORE: {overall_score:.2f}%")
print("="*55)

--- THÔNG SỐ ĐẦU VÀO HỆ THỐNG ---
 -> Kích thước tập dữ liệu THẬT (X_train_np): (1820, 93)
 -> Kích thước tập dữ liệu GIẢ   (X_synth):    (10000, 93)

   KẾT QUẢ KIỂM ĐỊNH PHÂN PHỐI DỮ LIỆU TỔNG HỢP HOÀN CHỈNH
1. Wasserstein Distance Metric (Đơn biến - Đã chuẩn hóa):
   - Khoảng cách thô định lượng: 0.4287
   - Độ tin cậy hình thái cột   : 65.14%
     (Tránh lỗi tràn số mũ của dữ liệu trắc lượng thô)

2. Maximum Mean Discrepancy Metric (Đa biến toàn cục):
   - Khoảng cách thô tương quan: 0.0255
   - Độ tin cậy tương quan giải phẫu (Allometry): 97.48%

-------------------------------------------------------
--> OVERALL DISTRIBUTION TRUST SCORE: 81.31%


In [ ]:
!pip install sdmetrics

In [ ]:
from sdmetrics.reports.single_table import QualityReport

In [29]:
num_features = X_train_np.shape[1]
column_names = [f'feature_{i}' for i in range(num_features)]

# 1. Khởi tạo DataFrame cho dữ liệu THẬT (Gồm các đặc trưng xương + cột nhãn giới tính số nguyên)
real_df = pd.DataFrame(X_train_np, columns=column_names)
real_df['Gender'] = y_train_encoded_np

# 2. Khởi tạo DataFrame cho dữ liệu GIẢ (Gộp mảng male và female lại)
X_synth = np.vstack((male_generated, female_generated))

# Đồng bộ hóa nhãn số nguyên từ LabelEncoder 'le' của bạn
# (Giả định le.transform(['male'])[0] và le.transform(['female'])[0] trả về nhãn tương ứng)
labels_male_encoded = np.full(male_generated.shape[0], le.transform(['Male'])[0])
labels_female_encoded = np.full(female_generated.shape[0], le.transform(['Female'])[0])
y_synth_encoded = np.concatenate((labels_male_encoded, labels_female_encoded))

synth_df = pd.DataFrame(X_synth, columns=column_names)
synth_df['Gender'] = y_synth_encoded


# =====================================================================
# STEP 2: KHỞI TẠO METADATA THEO TIÊU CHUẨN SDMETRICS
# =====================================================================
# SDMetrics cần biết cột nào là số liên tục (numerical) và cột nào là phân loại (categorical)
metadata = {
    'columns': {col: {'sdtype': 'numerical'} for col in column_names},
}
# Bổ sung thông tin cột gender là biến phân loại (nhãn encoded số nguyên)
metadata['columns']['Gender'] = {'sdtype': 'categorical'}


# =====================================================================
# STEP 3: CHẠY BÁO CÁO CHẤT LƯỢNG VÀ TRÍCH XUẤT SỐ % ĐỘ TIN CẬY
# =====================================================================
print("--- ĐANG TIẾN HÀNH CHẤM ĐIỂM VỚI SDMETRICS ---")

# Khởi tạo và tạo báo cáo
report = QualityReport()
report.generate(real_df, synth_df, metadata)

# 1. Lấy điểm số phần trăm tổng quát (Overall Score)
overall_score_pct = report.get_score() * 100

# 2. Lấy điểm số chi tiết theo từng khía cạnh cốt lõi
properties_df = report.get_properties()

print("\n" + "="*55)
print("   BÁO CÁO CHẤT LƯỢNG DỮ LIỆU TỔNG HỢP TỪ SDMETRICS")
print("="*55)
for index, row in properties_df.iterrows():
    property_name = row['Property']
    score_pct = row['Score'] * 100

    # Dịch nghĩa các thuộc tính của SDMetrics sang ngôn ngữ luận án SYNOS
    if property_name == 'Column Shapes':
        print(f"1. Độ tương đồng Hình dáng Đơn biến (Column Shapes): {score_pct:.2f}%")
        print("   (Bao gồm kiểm định KS-Test cho biến số và Chi-Square cho giới tính)")
    elif property_name == 'Column Pair Trends':
        print(f"\n2. Độ tương đồng Tương quan Đa biến (Column Pair Trends): {score_pct:.2f}%")
        print("   (Kiểm tra ma trận tương quan chéo giữ luật giải phẫu Allometry)")

print("-" * 55)
print(f"--> OVERALL SDMETRICS TRUST SCORE: {overall_score_pct:.2f}%")
print("="*55)

--- ĐANG TIẾN HÀNH CHẤM ĐIỂM VỚI SDMETRICS ---
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 94/94 [00:13<00:00,  6.93it/s]|
Column Shapes Score: 81.33%

(2/2) Evaluating Column Pair Trends: |██████████| 4371/4371 [00:52<00:00, 82.68it/s]| 
Column Pair Trends Score: 96.77%

Overall Score (Average): 89.05%


   BÁO CÁO CHẤT LƯỢNG DỮ LIỆU TỔNG HỢP TỪ SDMETRICS
1. Độ tương đồng Hình dáng Đơn biến (Column Shapes): 81.33%
   (Bao gồm kiểm định KS-Test cho biến số và Chi-Square cho giới tính)

2. Độ tương đồng Tương quan Đa biến (Column Pair Trends): 96.77%
   (Kiểm tra ma trận tương quan chéo giữ luật giải phẫu Allometry)
-------------------------------------------------------
--> OVERALL SDMETRICS TRUST SCORE: 89.05%


In [30]:

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# =====================================================================
# DATA PREPROCESSING & DATASET SYNCHRONISATION
# =====================================================================
target_col = 'Gender'
label_mapping = {'Female': 0, 'Male': 1}

def extract_features_labels(df, target, mapping=None):
    df_proc = df.copy()
    if mapping:
        df_proc[target] = df_proc[target].map(mapping)
    return df_proc.drop(columns=[target]).values, df_proc[target].values.astype(int)

# Extract real and synthetic experimental matrices
X_train_real, y_train_real = extract_features_labels(df_train, target_col, label_mapping)
X_val_real, y_val_real = extract_features_labels(df_val, target_col, label_mapping)
X_train_synth, y_train_synth = extract_features_labels(synth_df, target_col)

# =====================================================================
# MODEL ARCHITECTURE CONFIGURATION
# =====================================================================
models_config = {
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=5, random_state=42, eval_metric='logloss'),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42)
}

# =====================================================================
# EVALUATION EXECUTION (Baseline TRTR vs. Generative TSTR)
# =====================================================================
report_data = []

def evaluate_pipeline(model, X_train, y_train, X_val):
    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    # Elegant extraction of probability or decision scores for AUROC calculation
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_val)[:, 1]
    else:
        probs = getattr(model, "decision_function", lambda x: 0)(X_val)
    return preds, probs

for name, model in models_config.items():
    # 1. Baseline Configuration (Train Real, Test Real)
    y_pred_real, y_prob_real = evaluate_pipeline(model, X_train_real, y_train_real, X_val_real)

    # 2. Synthetic Configuration (Train Synthetic, Test Real)
    y_pred_synth, y_prob_synth = evaluate_pipeline(model, X_train_synth, y_train_synth, X_val_real)

    report_data.append({
        "Model": name,
        "Acc_Real": accuracy_score(y_val_real, y_pred_real),
        "F1_Real": f1_score(y_val_real, y_pred_real, average='macro'),
        "AUC_Real": roc_auc_score(y_val_real, y_prob_real),
        "Acc_Synth": accuracy_score(y_val_real, y_pred_synth),
        "F1_Synth": f1_score(y_val_real, y_pred_synth, average='macro'),
        "AUC_Synth": roc_auc_score(y_val_real, y_prob_synth)
    })

# =====================================================================
# PERFORMANCE COMPILATION & GENERATIVE EFFICACY ANALYSIS
# =====================================================================
df_res = pd.DataFrame(report_data).set_index("Model")

for metric in ['Acc', 'F1', 'AUC']:
    df_res[f'{metric}_Efficacy (%)'] = (df_res[f'{metric}_Synth'] / df_res[f'{metric}_Real']) * 100

print("\n" + "="*85)
print("             BẢNG TỔNG HỢP HIỆU NĂNG ĐỘ TIN CẬY (TSTR vs TRTR)")
print("="*85)

format_dict = {col: '{:,.2f}%'.format for col in df_res.columns if 'Efficacy' in col}
other_cols = {col: '{:,.4f}'.format for col in df_res.columns if 'Efficacy' not in col}

print(df_res.style.format({**format_dict, **other_cols}).to_string())
print("="*85)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



             BẢNG TỔNG HỢP HIỆU NĂNG ĐỘ TIN CẬY (TSTR vs TRTR)
 Acc_Real F1_Real AUC_Real Acc_Synth F1_Synth AUC_Synth Acc_Efficacy (%) F1_Efficacy (%) AUC_Efficacy (%)
Model         
XGBoost 0.9922 0.9912 0.9996 0.9317 0.9190 0.9840 93.90% 92.72% 98.44%
Random Forest 0.9809 0.9782 0.9982 0.9157 0.8989 0.9785 93.35% 91.90% 98.02%
Logistic Regression 0.9995 0.9995 1.0000 0.8870 0.8580 0.9845 88.74% 85.85% 98.45%
SVM 0.9729 0.9690 0.9969 0.9543 0.9472 0.9934 98.09% 97.75% 99.65%



In [ ]:
df_res

In [ ]:
# Kiểm tra xem dữ liệu giả có bị trùng với dữ liệu thật không
# Sử dụng real_df thay vì df_train vì real_df có cùng cấu trúc cột và kiểu dữ liệu (đặc biệt là 'Gender') với synth_df
duplicates = pd.merge(synth_df, real_df, how='inner')
print(f"Số lượng bản ghi bị sao chép y hệt: {len(duplicates)}")

In [ ]:
# 1. Kiểm tra sự trùng lặp giữa Train và Val
intersection = pd.merge(df_train, df_val, how='inner')
print(f"Số lượng bản ghi trùng lặp giữa Train và Val: {len(intersection)}")

# 2. Kiểm tra các cột có tính chất định danh
# Tìm các cột có số lượng giá trị duy nhất cao bằng số lượng mẫu
for col in df_train.columns:
    if df_train[col].nunique() == len(df_train):
        print(f"Cảnh báo: Cột '{col}' có thể là biến ID gây rò rỉ dữ liệu!")